# FoodSure AI — Robustness & Cross-Validation

## Evaluating Model Stability Across Multiple Data Splits

Previous FoodSure AI experiments used a fixed 80/20 train-test split:

- 280 training samples
- 70 test samples

This provided a consistent benchmark for comparing models and modalities.

However, performance on a single fixed test set may not fully represent
model stability.

This notebook therefore evaluates selected models using repeated
cross-validation across multiple train-validation splits.

### Objectives

- evaluate classification stability
- evaluate regression stability
- measure variation across folds
- compare cross-validation results with the fixed test-set results
- investigate class-wise performance
- identify potential generalization limitations

### Leakage Prevention

All preprocessing operations that learn parameters from the data, such as
feature scaling and PCA, must be fitted separately within each
cross-validation training fold.

This prevents information from the validation fold from influencing the
training process.

### Important Limitation

Cross-validation evaluates stability within the available dataset.

It does not guarantee performance on independently collected real-world
turmeric samples.

The current dataset contains controlled turmeric samples adulterated with
wheat flour.

## 1. Import Required Libraries

This section imports the libraries required for cross-validation, model evaluation, and statistical comparison of model performance.

We will use stratified cross-validation for classification tasks so that each fold maintains a similar distribution of adulteration classes.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, KFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

## 2. Load Training Data for Cross-Validation

Cross-validation is performed only on the training portion of the dataset.

The fixed 70-sample test set is kept completely separate and is not used during cross-validation.

This ensures that the test set remains an independent benchmark for final model evaluation.

In [ ]:
preprocessed_dir = "../preprocessed"

X_color_train = pd.read_csv(
    f"{preprocessed_dir}/X_color_train_scaled.csv"
)

X_ftir_train = pd.read_csv(
    f"{preprocessed_dir}/X_ftir_train_scaled.csv"
)

X_image_train = pd.read_csv(
    f"{preprocessed_dir}/X_image_train_scaled.csv"
)

y_train = pd.read_csv(
    f"{preprocessed_dir}/y_train.csv"
).squeeze()

print("Color:", X_color_train.shape)
print("FTIR:", X_ftir_train.shape)
print("Image:", X_image_train.shape)
print("Target:", y_train.shape)

## 3. Cross-Validation Strategy

We use 5-fold stratified cross-validation for classification.

The 280 training samples are divided into five folds. In each iteration:

- 4 folds are used for training
- 1 fold is used for validation
- the process is repeated five times
- every training sample is used for validation exactly once

Stratification is used to preserve the distribution of the seven adulteration classes across the folds.

The fixed 70-sample test set remains completely untouched during this process.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Number of folds:", cv.n_splits)
print("Shuffle:", cv.shuffle)
print("Random state:", cv.random_state)

Number of folds: 5
Shuffle: True
Random state: 42


## 4. Evaluation Metrics

Each cross-validation fold is evaluated using:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score

Macro averaging gives equal importance to each adulteration class, which is useful when evaluating performance across all seven adulteration levels.

We will report both the mean and standard deviation across the five folds.

The mean represents average validation performance, while the standard deviation indicates how much the performance varies across different folds.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

print("Evaluation metrics:")
for metric in scoring:
    print("-", metric)

Evaluation metrics:
- accuracy
- precision_macro
- recall_macro
- f1_macro


## 5. Color + Logistic Regression

Logistic Regression is evaluated on the Color modality using 5-fold stratified cross-validation.

The purpose is to determine whether the performance observed in the fixed train-test experiment remains reasonably stable across different training-validation splits.

The evaluation reports the mean and standard deviation of Accuracy, Macro Precision, Macro Recall, and Macro F1-score across the five folds.

In [ ]:
color_lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

color_lr_cv = cross_validate(
    color_lr,
    X_color_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

color_lr_cv_results = {
    "Accuracy": color_lr_cv["test_accuracy"],
    "Macro Precision": color_lr_cv["test_precision_macro"],
    "Macro Recall": color_lr_cv["test_recall_macro"],
    "Macro F1": color_lr_cv["test_f1_macro"]
}

for metric, scores in color_lr_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [0.9821 1.     0.9821 0.9464 0.9821]
  Mean: 0.9786
  Std:  0.0175

Macro Precision:
  Fold scores: [0.9841 1.     0.9841 0.9556 0.9841]
  Mean: 0.9816
  Std:  0.0144

Macro Recall:
  Fold scores: [0.9821 1.     0.9821 0.9464 0.9821]
  Mean: 0.9786
  Std:  0.0175

Macro F1:
  Fold scores: [0.9821 1.     0.9821 0.9458 0.9821]
  Mean: 0.9784
  Std:  0.0177



### Interpretation

Color + Logistic Regression achieved high average validation performance across the five folds.

The mean accuracy was 97.86%, with a standard deviation of 1.75 percentage points. Macro Precision, Macro Recall, and Macro F1 were also close to 98%.

The relatively small variation across folds suggests that the model's performance is reasonably stable within the available training dataset.

However, these results are based only on cross-validation of the training data. They should not be treated as the final test-set performance or as evidence of guaranteed performance on independently collected samples.

## 6. Color + SVM

The RBF-kernel Support Vector Machine is evaluated using the same 5-fold stratified cross-validation procedure.

Using the same cross-validation strategy allows the stability of the SVM to be compared with the Logistic Regression baseline under consistent conditions.

In [ ]:
color_svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

color_svm_cv = cross_validate(
    color_svm,
    X_color_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

color_svm_cv_results = {
    "Accuracy": color_svm_cv["test_accuracy"],
    "Macro Precision": color_svm_cv["test_precision_macro"],
    "Macro Recall": color_svm_cv["test_recall_macro"],
    "Macro F1": color_svm_cv["test_f1_macro"]
}

for metric, scores in color_svm_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [0.9643 0.9821 0.9464 0.875  0.9286]
  Mean: 0.9393
  Std:  0.0368

Macro Precision:
  Fold scores: [0.9683 0.9841 0.9556 0.8744 0.9397]
  Mean: 0.9444
  Std:  0.0379

Macro Recall:
  Fold scores: [0.9643 0.9821 0.9464 0.875  0.9286]
  Mean: 0.9393
  Std:  0.0368

Macro F1:
  Fold scores: [0.9641 0.9821 0.9458 0.8737 0.9265]
  Mean: 0.9385
  Std:  0.0373



### Interpretation

Color + SVM achieved a mean cross-validation accuracy of 93.93%, with a standard deviation of 3.68 percentage points.

The fold accuracy ranged from 87.50% to 98.21%, indicating noticeable variation across the validation folds.

Compared with the Color + Logistic Regression experiment, the SVM showed greater fold-to-fold variation. Therefore, its performance appears less stable across different training-validation splits within the available dataset.

These results demonstrate why cross-validation is useful in addition to evaluation on a single fixed test set.

## 7. Color + Random Forest

Random Forest is evaluated using the same 5-fold stratified cross-validation procedure.

This experiment provides a robustness estimate for the tree-based Color baseline and allows its fold-to-fold variation to be compared with Logistic Regression and SVM.

In [ ]:
color_rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

color_rf_cv = cross_validate(
    color_rf,
    X_color_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

color_rf_cv_results = {
    "Accuracy": color_rf_cv["test_accuracy"],
    "Macro Precision": color_rf_cv["test_precision_macro"],
    "Macro Recall": color_rf_cv["test_recall_macro"],
    "Macro F1": color_rf_cv["test_f1_macro"]
}

for metric, scores in color_rf_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [0.9821 0.9821 0.9464 0.9286 0.8571]
  Mean: 0.9393
  Std:  0.0460

Macro Precision:
  Fold scores: [0.9841 0.9841 0.9524 0.9357 0.877 ]
  Mean: 0.9467
  Std:  0.0395

Macro Recall:
  Fold scores: [0.9821 0.9821 0.9464 0.9286 0.8571]
  Mean: 0.9393
  Std:  0.0460

Macro F1:
  Fold scores: [0.9821 0.9821 0.9449 0.928  0.8562]
  Mean: 0.9387
  Std:  0.0463



## 8. Color Modality — Cross-Validation Comparison

The three Color-based classification models are compared using the mean and standard deviation obtained from 5-fold stratified cross-validation.

The mean represents average validation performance, while the standard deviation indicates fold-to-fold variation.

In [ ]:
color_cv_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest"
    ],
    "Accuracy Mean": [
        color_lr_cv_results["Accuracy"].mean(),
        color_svm_cv_results["Accuracy"].mean(),
        color_rf_cv_results["Accuracy"].mean()
    ],
    "Accuracy Std": [
        color_lr_cv_results["Accuracy"].std(),
        color_svm_cv_results["Accuracy"].std(),
        color_rf_cv_results["Accuracy"].std()
    ],
    "Macro F1 Mean": [
        color_lr_cv_results["Macro F1"].mean(),
        color_svm_cv_results["Macro F1"].mean(),
        color_rf_cv_results["Macro F1"].mean()
    ],
    "Macro F1 Std": [
        color_lr_cv_results["Macro F1"].std(),
        color_svm_cv_results["Macro F1"].std(),
        color_rf_cv_results["Macro F1"].std()
    ]
})

color_cv_summary.round(4)

,Model,Accuracy Mean,Accuracy Std,Macro F1 Mean,Macro F1 Std
0,Logistic Regression,0.9786,0.0175,0.9784,0.0177
1,SVM (RBF),0.9393,0.0368,0.9385,0.0373
2,Random Forest,0.9393,0.0460,0.9387,0.0463


### Interpretation

The Color + Logistic Regression model showed the highest mean cross-validation performance among the three Color baselines, with an accuracy of 97.86% and a Macro F1-score of 97.84%.

It also showed the lowest accuracy variation across folds, with a standard deviation of 1.75 percentage points.

The Color + SVM and Color + Random Forest models both achieved lower mean accuracy of 93.93%. Their larger standard deviations indicate greater variation across the validation folds.

These results suggest that Logistic Regression provides more consistent performance across the available training data for the Color modality.

However, this comparison is limited to the current dataset and cross-validation procedure. It does not establish universal superiority or guarantee performance on independently collected samples.

-------------------------------------------------------

## 9. FTIR + Logistic Regression

FTIR spectroscopy contains 2522 spectral features, making dimensionality reduction useful before classification.

For cross-validation, scaling and PCA are placed inside a pipeline so that both transformations are fitted independently within each training fold.

This prevents information from the validation fold from influencing the preprocessing steps.

The Logistic Regression classifier is then trained on the fold-specific PCA representation.

In [ ]:
from sklearn.decomposition import PCA
ftir_lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

ftir_lr_cv = cross_validate(
    ftir_lr_pipeline,
    X_ftir_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

ftir_lr_cv_results = {
    "Accuracy": ftir_lr_cv["test_accuracy"],
    "Macro Precision": ftir_lr_cv["test_precision_macro"],
    "Macro Recall": ftir_lr_cv["test_recall_macro"],
    "Macro F1": ftir_lr_cv["test_f1_macro"]
}

for metric, scores in ftir_lr_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Precision:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Recall:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro F1:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000



### Interpretation

FTIR + Logistic Regression achieved 100% accuracy, Macro Precision, Macro Recall, and Macro F1 across all five cross-validation folds.

The standard deviation was 0 for every metric, indicating identical performance across the five validation folds.

This indicates extremely strong separability of the adulteration classes within the available training dataset.

However, perfect cross-validation performance should be interpreted cautiously. It does not guarantee equivalent performance on independently collected turmeric samples. The result may reflect the controlled nature of the current dataset and the spectral information available in the measured samples.

Further evaluation using the untouched test set, independent samples, and robustness analysis is therefore important before drawing conclusions about real-world generalization.

## 10. FTIR + SVM

The RBF-kernel Support Vector Machine is evaluated on the FTIR modality using the same leakage-safe pipeline.

Scaling and PCA are fitted independently within each cross-validation training fold before the SVM is trained.

In [ ]:
ftir_svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"
    ))
])

ftir_svm_cv = cross_validate(
    ftir_svm_pipeline,
    X_ftir_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

ftir_svm_cv_results = {
    "Accuracy": ftir_svm_cv["test_accuracy"],
    "Macro Precision": ftir_svm_cv["test_precision_macro"],
    "Macro Recall": ftir_svm_cv["test_recall_macro"],
    "Macro F1": ftir_svm_cv["test_f1_macro"]
}

for metric, scores in ftir_svm_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Precision:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Recall:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro F1:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000



### Interpretation

FTIR + SVM also achieved 100% across all five cross-validation folds for Accuracy, Macro Precision, Macro Recall, and Macro F1.

The zero standard deviation indicates identical performance across all folds.

Together with the FTIR + Logistic Regression result, this suggests that the FTIR representation contains highly discriminative information for separating the seven adulteration levels within the available dataset.

However, the perfect cross-validation performance should be interpreted cautiously. It reflects performance within the current controlled dataset and does not guarantee equivalent performance on independently collected real-world turmeric samples.

The result will therefore be compared with the fixed test-set evaluation and other modalities before drawing broader conclusions.

## 11. FTIR + Random Forest

Random Forest is evaluated on the FTIR modality using the same leakage-safe cross-validation pipeline.

Scaling and PCA are fitted independently within each training fold before the Random Forest classifier is trained.

In [ ]:
ftir_rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

ftir_rf_cv = cross_validate(
    ftir_rf_pipeline,
    X_ftir_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

ftir_rf_cv_results = {
    "Accuracy": ftir_rf_cv["test_accuracy"],
    "Macro Precision": ftir_rf_cv["test_precision_macro"],
    "Macro Recall": ftir_rf_cv["test_recall_macro"],
    "Macro F1": ftir_rf_cv["test_f1_macro"]
}

for metric, scores in ftir_rf_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Precision:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro Recall:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000

Macro F1:
  Fold scores: [1. 1. 1. 1. 1.]
  Mean: 1.0000
  Std:  0.0000



### Interpretation

FTIR + Random Forest achieved 100% Accuracy, Macro Precision,
Macro Recall, and Macro F1 across all five cross-validation folds.

The standard deviation was 0 for every metric, indicating identical
performance across all validation folds.

Together with the FTIR + Logistic Regression and FTIR + SVM results,
the three FTIR models all achieved perfect cross-validation performance
on the available training dataset.

This indicates that the FTIR representation contains highly
discriminative information for separating the seven adulteration
levels within this controlled dataset.

However, perfect cross-validation performance should be interpreted
cautiously. It does not guarantee equivalent performance on
independently collected turmeric samples. The dataset contains
controlled samples with specific adulteration levels and a single
adulterant.

The FTIR results should therefore be compared with the untouched
test-set results and robustness experiments before making broader
claims about generalization.

-----------------------------------------------------------------------------

## 12. FTIR Modality — Cross-Validation Comparison

The three FTIR-based classification models are compared using the
mean and standard deviation obtained from 5-fold stratified
cross-validation.

The mean represents average validation performance, while the
standard deviation indicates variation across the validation folds.

In [ ]:
ftir_cv_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest"
    ],
    "Accuracy Mean": [
        ftir_lr_cv_results["Accuracy"].mean(),
        ftir_svm_cv_results["Accuracy"].mean(),
        ftir_rf_cv_results["Accuracy"].mean()
    ],
    "Accuracy Std": [
        ftir_lr_cv_results["Accuracy"].std(),
        ftir_svm_cv_results["Accuracy"].std(),
        ftir_rf_cv_results["Accuracy"].std()
    ],
    "Macro F1 Mean": [
        ftir_lr_cv_results["Macro F1"].mean(),
        ftir_svm_cv_results["Macro F1"].mean(),
        ftir_rf_cv_results["Macro F1"].mean()
    ],
    "Macro F1 Std": [
        ftir_lr_cv_results["Macro F1"].std(),
        ftir_svm_cv_results["Macro F1"].std(),
        ftir_rf_cv_results["Macro F1"].std()
    ]
})

ftir_cv_summary.round(4)

,Model,Accuracy Mean,Accuracy Std,Macro F1 Mean,Macro F1 Std
0,Logistic Regression,1.0,0.0,1.0,0.0
1,SVM (RBF),1.0,0.0,1.0,0.0
2,Random Forest,1.0,0.0,1.0,0.0


### Interpretation

All three FTIR-based models achieved identical cross-validation
performance on the available training dataset.

Logistic Regression, SVM (RBF), and Random Forest each achieved:

- 100% mean Accuracy
- 100% mean Macro F1
- 0% Accuracy standard deviation
- 0% Macro F1 standard deviation

This means that all five validation folds produced perfect
classification performance for each of the three models.

The consistency across different model families suggests that the
current FTIR representation provides highly discriminative information
for separating the seven adulteration levels in this controlled dataset.

However, the perfect cross-validation results should not be interpreted
as proof of universal or real-world performance. The dataset contains
controlled samples with a single adulterant and predefined
concentration levels.

Independent samples and the untouched test set are still required to
evaluate generalization.

-----------------------------------------------------------------------------

## 13. Image + Logistic Regression

The Image modality is represented by 1280-dimensional EfficientNet-B0
features extracted from the turmeric images.

Because the feature space is relatively high-dimensional, PCA is applied
inside the cross-validation pipeline to reduce dimensionality while
retaining approximately 90% of the variance.

Both scaling and PCA are fitted independently within each training fold
to avoid information leakage.

In [ ]:
image_lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

image_lr_cv = cross_validate(
    image_lr_pipeline,
    X_image_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

image_lr_cv_results = {
    "Accuracy": image_lr_cv["test_accuracy"],
    "Macro Precision": image_lr_cv["test_precision_macro"],
    "Macro Recall": image_lr_cv["test_recall_macro"],
    "Macro F1": image_lr_cv["test_f1_macro"]
}

for metric, scores in image_lr_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [0.9464 0.9464 0.9643 0.9464 0.9821]
  Mean: 0.9571
  Std:  0.0143

Macro Precision:
  Fold scores: [0.9536 0.9556 0.9683 0.9536 0.9841]
  Mean: 0.9630
  Std:  0.0119

Macro Recall:
  Fold scores: [0.9464 0.9464 0.9643 0.9464 0.9821]
  Mean: 0.9571
  Std:  0.0143

Macro F1:
  Fold scores: [0.9472 0.9458 0.9641 0.9472 0.9821]
  Mean: 0.9573
  Std:  0.0141



### Interpretation

Image + Logistic Regression achieved a mean cross-validation accuracy
of 95.71% across the five stratified folds.

The mean Macro F1-score was 95.73%, while Macro Precision reached
96.30% and Macro Recall reached 95.71%.

The accuracy standard deviation was 1.43 percentage points, indicating
relatively small variation across the validation folds. The fold
accuracy ranged from 94.64% to 98.21%.

These results indicate that the EfficientNet-B0 image representation
contains useful information for distinguishing the seven adulteration
levels within the available dataset.

However, the results are based on cross-validation within the current
controlled dataset. They do not guarantee equivalent performance on
independently collected turmeric samples or different adulterants.

## 14. Image + SVM

The Image modality is evaluated using an RBF-kernel Support Vector
Machine.

Because the EfficientNet-B0 representation contains 1280 features,
StandardScaler and PCA are applied inside the cross-validation
pipeline before classification.

This ensures that the scaling and dimensionality-reduction steps are
learned independently within each training fold.

In [ ]:
image_svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"
    ))
])

image_svm_cv = cross_validate(
    image_svm_pipeline,
    X_image_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

image_svm_cv_results = {
    "Accuracy": image_svm_cv["test_accuracy"],
    "Macro Precision": image_svm_cv["test_precision_macro"],
    "Macro Recall": image_svm_cv["test_recall_macro"],
    "Macro F1": image_svm_cv["test_f1_macro"]
}

for metric, scores in image_svm_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

Accuracy:
  Fold scores: [0.9286 0.9464 0.9643 0.9643 0.9464]
  Mean: 0.9500
  Std:  0.0134

Macro Precision:
  Fold scores: [0.9377 0.9478 0.9683 0.9683 0.9556]
  Mean: 0.9555
  Std:  0.0118

Macro Recall:
  Fold scores: [0.9286 0.9464 0.9643 0.9643 0.9464]
  Mean: 0.9500
  Std:  0.0134

Macro F1:
  Fold scores: [0.9293 0.9462 0.9641 0.9641 0.9458]
  Mean: 0.9499
  Std:  0.0131



### Interpretation

Image + SVM achieved a mean cross-validation accuracy of 95.00%
across the five stratified folds.

The mean Macro F1-score was 94.99%, while Macro Precision reached
95.55% and Macro Recall reached 95.00%.

The accuracy standard deviation was 1.34 percentage points, indicating
relatively small variation across the validation folds. The fold
accuracy ranged from 92.86% to 96.43%.

Compared with Image + Logistic Regression, the SVM produced slightly
lower mean Accuracy and Macro F1 on the same cross-validation splits.

Both models nevertheless showed relatively consistent performance
across the available training data.

These results are specific to the current controlled dataset and do
not guarantee equivalent performance on independently collected
samples or different adulterants.

## 15. Image + Random Forest

The Image modality is evaluated using a Random Forest classifier.

Because the EfficientNet-B0 representation contains 1280 features,
StandardScaler and PCA are applied inside the cross-validation
pipeline before classification.

The complete preprocessing and classification pipeline is fitted
independently within each cross-validation fold.

In [ ]:
image_rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

image_rf_cv = cross_validate(
    image_rf_pipeline,
    X_image_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

image_rf_cv_results = {
    "Accuracy": image_rf_cv["test_accuracy"],
    "Macro Precision": image_rf_cv["test_precision_macro"],
    "Macro Recall": image_rf_cv["test_recall_macro"],
    "Macro F1": image_rf_cv["test_f1_macro"]
}

for metric, scores in image_rf_cv_results.items():
    print(f"{metric}:")
    print(f"  Fold scores: {np.round(scores, 4)}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std:  {scores.std():.4f}")
    print()

### Interpretation

Image + Random Forest achieved a mean cross-validation accuracy of
92.50% across the five stratified folds.

The mean Macro F1-score was 92.51%, while Macro Precision reached
93.24% and Macro Recall reached 92.50%.

The accuracy standard deviation was 2.86 percentage points, indicating
more fold-to-fold variation than the Image + Logistic Regression and
Image + SVM models.

The fold accuracy ranged from 89.29% to 96.43%.

Among the three Image-based models evaluated so far, the Random Forest
model showed lower mean validation performance and greater variation
across folds.

These results are specific to the current controlled dataset and do
not guarantee equivalent performance on independently collected
turmeric samples or different adulterants.

## 16. Image Modality — Cross-Validation Comparison

The three Image-based classification models are compared using the
mean and standard deviation obtained from 5-fold stratified
cross-validation.

The comparison helps evaluate both average validation performance and
fold-to-fold stability.

In [ ]:
image_cv_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest"
    ],
    "Accuracy Mean": [
        image_lr_cv_results["Accuracy"].mean(),
        image_svm_cv_results["Accuracy"].mean(),
        image_rf_cv_results["Accuracy"].mean()
    ],
    "Accuracy Std": [
        image_lr_cv_results["Accuracy"].std(),
        image_svm_cv_results["Accuracy"].std(),
        image_rf_cv_results["Accuracy"].std()
    ],
    "Macro F1 Mean": [
        image_lr_cv_results["Macro F1"].mean(),
        image_svm_cv_results["Macro F1"].mean(),
        image_rf_cv_results["Macro F1"].mean()
    ],
    "Macro F1 Std": [
        image_lr_cv_results["Macro F1"].std(),
        image_svm_cv_results["Macro F1"].std(),
        image_rf_cv_results["Macro F1"].std()
    ]
})

image_cv_summary.round(4)

,Model,Accuracy Mean,Accuracy Std,Macro F1 Mean,Macro F1 Std
0,Logistic Regression,0.9571,0.0143,0.9573,0.0141
1,SVM (RBF),0.9500,0.0134,0.9499,0.0131
2,Random Forest,0.9250,0.0286,0.9251,0.0287


### Interpretation

The Image modality showed relatively strong and consistent
cross-validation performance across the three models.

Image + Logistic Regression achieved a mean Accuracy of 95.71% and a
Macro F1-score of 95.73%, with an Accuracy standard deviation of
1.43 percentage points.

Image + SVM achieved a mean Accuracy of 95.00% and a Macro F1-score
of 94.99%. Its fold-to-fold variation was also relatively small.

Image + Random Forest achieved a lower mean Accuracy of 92.50% and
Macro F1-score of 92.51%, with a larger standard deviation of
2.86 percentage points.

Overall, the three models demonstrate that the EfficientNet-B0 image
representation contains useful information for distinguishing the
adulteration levels in the current dataset.

However, these results represent validation performance within the
available controlled dataset and should not be interpreted as
guaranteed real-world performance.

--------------------------------------------------------

## 17. Overall Cross-Validation Comparison

This section combines the cross-validation results obtained for the
Color, FTIR, and Image modalities.

The comparison considers:

- Accuracy
- Macro F1-score
- Fold-to-fold standard deviation

The purpose is to understand how consistently each modality performs
across different validation splits.

These results are based on the current cross-validation setup and will
be revisited after applying the strict leakage-safe preprocessing
pipeline.

In [ ]:
overall_cv_summary = pd.DataFrame({
    "Modality": [
        "Color",
        "Color",
        "Color",
        "FTIR",
        "FTIR",
        "FTIR",
        "Image",
        "Image",
        "Image"
    ],
    "Model": [
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest",
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest",
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest"
    ],
    "Accuracy Mean": [
        color_lr_cv_results["Accuracy"].mean(),
        color_svm_cv_results["Accuracy"].mean(),
        color_rf_cv_results["Accuracy"].mean(),
        ftir_lr_cv_results["Accuracy"].mean(),
        ftir_svm_cv_results["Accuracy"].mean(),
        ftir_rf_cv_results["Accuracy"].mean(),
        image_lr_cv_results["Accuracy"].mean(),
        image_svm_cv_results["Accuracy"].mean(),
        image_rf_cv_results["Accuracy"].mean()
    ],
    "Accuracy Std": [
        color_lr_cv_results["Accuracy"].std(),
        color_svm_cv_results["Accuracy"].std(),
        color_rf_cv_results["Accuracy"].std(),
        ftir_lr_cv_results["Accuracy"].std(),
        ftir_svm_cv_results["Accuracy"].std(),
        ftir_rf_cv_results["Accuracy"].std(),
        image_lr_cv_results["Accuracy"].std(),
        image_svm_cv_results["Accuracy"].std(),
        image_rf_cv_results["Accuracy"].std()
    ],
    "Macro F1 Mean": [
        color_lr_cv_results["Macro F1"].mean(),
        color_svm_cv_results["Macro F1"].mean(),
        color_rf_cv_results["Macro F1"].mean(),
        ftir_lr_cv_results["Macro F1"].mean(),
        ftir_svm_cv_results["Macro F1"].mean(),
        ftir_rf_cv_results["Macro F1"].mean(),
        image_lr_cv_results["Macro F1"].mean(),
        image_svm_cv_results["Macro F1"].mean(),
        image_rf_cv_results["Macro F1"].mean()
    ],
    "Macro F1 Std": [
        color_lr_cv_results["Macro F1"].std(),
        color_svm_cv_results["Macro F1"].std(),
        color_rf_cv_results["Macro F1"].std(),
        ftir_lr_cv_results["Macro F1"].std(),
        ftir_svm_cv_results["Macro F1"].std(),
        ftir_rf_cv_results["Macro F1"].std(),
        image_lr_cv_results["Macro F1"].std(),
        image_svm_cv_results["Macro F1"].std(),
        image_rf_cv_results["Macro F1"].std()
    ]
})

overall_cv_summary.round(4)

,Modality,Model,Accuracy Mean,Accuracy Std,Macro F1 Mean,Macro F1 Std
0,Color,Logistic Regression,0.9786,0.0175,0.9784,0.0177
1,Color,SVM (RBF),0.9393,0.0368,0.9385,0.0373
2,Color,Random Forest,0.9393,0.0460,0.9387,0.0463
3,FTIR,Logistic Regression,1.0000,0.0000,1.0000,0.0000
4,FTIR,SVM (RBF),1.0000,0.0000,1.0000,0.0000
5,FTIR,Random Forest,1.0000,0.0000,1.0000,0.0000
6,Image,Logistic Regression,0.9571,0.0143,0.9573,0.0141
7,Image,SVM (RBF),0.9500,0.0134,0.9499,0.0131
8,Image,Random Forest,0.9250,0.0286,0.9251,0.0287


## 18. Cross-Validation vs Fixed Test-Set Results

Cross-validation provides an estimate of model performance across
multiple training-validation splits, while the fixed test set provides
a single independent evaluation using the 70 samples that were kept
untouched during model development.

This section compares the cross-validation mean performance with the
previously obtained fixed test-set performance.

The comparison helps identify whether model performance is reasonably
consistent across different evaluation settings.

The current cross-validation results should be interpreted with the
preprocessing-leakage limitation documented earlier and will be
revisited after the strict leakage-safe rerun.

In [ ]:
cv_vs_test_summary = pd.DataFrame({
    "Modality": [
        "Color",
        "Color",
        "Color",
        "FTIR",
        "FTIR",
        "FTIR",
        "Image",
        "Image",
        "Image"
    ],
    "Model": [
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest",
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest",
        "Logistic Regression",
        "SVM (RBF)",
        "Random Forest"
    ],
    "CV Accuracy": [
        color_lr_cv_results["Accuracy"].mean(),
        color_svm_cv_results["Accuracy"].mean(),
        color_rf_cv_results["Accuracy"].mean(),
        ftir_lr_cv_results["Accuracy"].mean(),
        ftir_svm_cv_results["Accuracy"].mean(),
        ftir_rf_cv_results["Accuracy"].mean(),
        image_lr_cv_results["Accuracy"].mean(),
        image_svm_cv_results["Accuracy"].mean(),
        image_rf_cv_results["Accuracy"].mean()
    ],
    "Fixed Test Accuracy": [
        0.8714,
        0.8857,
        0.8143,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan
    ],
    "CV Macro F1": [
        color_lr_cv_results["Macro F1"].mean(),
        color_svm_cv_results["Macro F1"].mean(),
        color_rf_cv_results["Macro F1"].mean(),
        ftir_lr_cv_results["Macro F1"].mean(),
        ftir_svm_cv_results["Macro F1"].mean(),
        ftir_rf_cv_results["Macro F1"].mean(),
        image_lr_cv_results["Macro F1"].mean(),
        image_svm_cv_results["Macro F1"].mean(),
        image_rf_cv_results["Macro F1"].mean()
    ]
})

cv_vs_test_summary.round(4)

,Modality,Model,CV Accuracy,Fixed Test Accuracy,CV Macro F1
0,Color,Logistic Regression,0.9786,0.8714,0.9784
1,Color,SVM (RBF),0.9393,0.8857,0.9385
2,Color,Random Forest,0.9393,0.8143,0.9387
3,FTIR,Logistic Regression,1.0000,NaN,1.0000
4,FTIR,SVM (RBF),1.0000,NaN,1.0000
5,FTIR,Random Forest,1.0000,NaN,1.0000
6,Image,Logistic Regression,0.9571,NaN,0.9573
7,Image,SVM (RBF),0.9500,NaN,0.9499
8,Image,Random Forest,0.9250,NaN,0.9251


### Interpretation

For the Color modality, cross-validation performance was higher than
the previously observed fixed test-set performance for all three
models.

Color + Logistic Regression achieved 97.86% mean cross-validation
accuracy compared with 87.14% on the fixed test set.

Color + SVM achieved 93.93% mean cross-validation accuracy compared
with 88.57% on the fixed test set.

Color + Random Forest achieved 93.93% mean cross-validation accuracy
compared with 81.43% on the fixed test set.

This difference shows that performance can vary depending on the
particular samples used for validation and testing.

Fixed-test results for FTIR and Image models are not included in this
comparison yet and are intentionally left as missing values rather
than being estimated.

The cross-validation results will be interpreted together with the
fixed test-set results and the strict leakage-safe rerun before
drawing final conclusions.

------------------------------------------------------------------------------

## 19. Class-wise Robustness Analysis

Overall accuracy and Macro F1 provide a summary of model performance,
but they do not show how individual adulteration levels behave.

This section examines class-wise performance using out-of-fold
predictions.

Each sample receives a prediction from a model that was trained
without that sample, providing a cross-validation-based estimate of
class-wise performance.

Precision, recall, and F1-score are examined for each adulteration
level.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

In [ ]:
color_lr_oof_pred = cross_val_predict(
    color_lr,
    X_color_train,
    y_train,
    cv=cv,
    method="predict"
)

color_lr_class_report = classification_report(
    y_train,
    color_lr_oof_pred,
    output_dict=True
)

color_lr_class_report_df = pd.DataFrame(
    color_lr_class_report
).T

color_lr_class_report_df.round(4)

,precision,recall,f1-score,support
0.0,1.0000,0.9750,0.9873,40.0000
1.0,0.9750,0.9750,0.9750,40.0000
2.0,0.9512,0.9750,0.9630,40.0000
3.0,1.0000,1.0000,1.0000,40.0000
5.0,1.0000,1.0000,1.0000,40.0000
7.0,0.9744,0.9500,0.9620,40.0000
9.0,0.9512,0.9750,0.9630,40.0000
accuracy,0.9786,0.9786,0.9786,0.9786
macro avg,0.9788,0.9786,0.9786,280.0000
weighted avg,0.9788,0.9786,0.9786,280.0000


### Interpretation

The class-wise out-of-fold results show consistently high performance
across all seven adulteration levels.

The 3% and 5% adulteration classes achieved perfect precision, recall,
and F1-score in the out-of-fold predictions.

The 0% class achieved a recall of 97.50%, while the 1% class achieved
97.50% for both precision and recall.

The 2%, 7%, and 9% classes showed slightly lower F1-scores, ranging
from 96.20% to 96.30%. These differences indicate that some
misclassification occurs between certain adulteration levels even
though the overall performance remains high.

In particular, the lower performance for 2%, 7%, and 9% suggests that
these levels may be relatively more challenging for the Color modality
under the current validation setup.

These observations are specific to the current dataset and model and
should not be interpreted as evidence that these adulteration levels
are universally harder to detect.

## 20. Overall Robustness Findings

The cross-validation experiments evaluated three modalities:

- Color measurements
- FTIR spectroscopy
- EfficientNet-B0 image features

Each modality was evaluated using Logistic Regression, SVM (RBF),
and Random Forest.

The analysis considered both average validation performance and
variation across folds.

The results provide an initial assessment of model stability within
the available controlled dataset.

In [ ]:
overall_robustness = pd.DataFrame({
    "Modality": [
        "Color",
        "FTIR",
        "Image"
    ],
    "Best Mean Accuracy": [
        color_cv_summary["Accuracy Mean"].max(),
        1.0000,
        image_cv_summary["Accuracy Mean"].max()
    ],
    "Corresponding Accuracy Std": [
        color_cv_summary.loc[
            color_cv_summary["Accuracy Mean"].idxmax(),
            "Accuracy Std"
        ],
        0.0000,
        image_cv_summary.loc[
            image_cv_summary["Accuracy Mean"].idxmax(),
            "Accuracy Std"
        ]
    ],
    "Best Mean Macro F1": [
        color_cv_summary["Macro F1 Mean"].max(),
        1.0000,
        image_cv_summary["Macro F1 Mean"].max()
    ]
})

overall_robustness.round(4)

,Modality,Best Mean Accuracy,Corresponding Accuracy Std,Best Mean Macro F1
0,Color,0.9786,0.0175,0.9784
1,FTIR,1.0000,0.0000,1.0000
2,Image,0.9571,0.0143,0.9573


## 21. Limitations

Several limitations should be considered when interpreting the
robustness results.

### 1. Controlled Dataset

The experiments use a controlled dataset containing turmeric samples
with predefined adulteration levels.

### 2. Single Adulterant

The current dataset uses wheat flour as the adulterant. Therefore,
performance against other possible adulterants has not been established.

### 3. Limited Sample Size

The dataset contains 350 samples distributed across seven adulteration
levels. Larger and more diverse datasets would provide stronger
evidence of generalization.

### 4. Cross-Validation Scope

Cross-validation evaluates stability within the available dataset. It
does not guarantee performance on independently collected real-world
samples.

### 5. Preprocessing Leakage Correction

The current cross-validation experiments used representations that were
initially scaled during the preprocessing stage. A strict
leakage-safe rerun, where scaling and dimensionality reduction are
fitted independently within each cross-validation training fold, is
required before the final robustness results are reported.

### 6. External Validation

No independently collected external dataset has been used for
validation in the current experiments.

Therefore, the system should be considered an AI-assisted research
prototype for turmeric adulteration screening and estimation rather
than a laboratory replacement or certified testing system.

## 22. Conclusion

This notebook evaluated the stability of classification models across
multiple cross-validation folds for the Color, FTIR, and Image
modalities.

The experiments demonstrated strong validation performance across the
three modalities on the current controlled dataset.

Class-wise analysis also showed consistently high performance across
the seven adulteration levels, with some variation between individual
classes.

However, cross-validation performance alone is not sufficient to
establish real-world generalization.

Before finalizing the robustness results, the experiments will be
repeated using a strictly leakage-safe preprocessing pipeline in which
all learned preprocessing steps are fitted independently within each
cross-validation training fold.

The final robustness assessment will therefore combine:

- leakage-safe cross-validation
- fixed test-set evaluation
- class-wise analysis
- comparison across modalities
- evaluation of model stability

The FoodSure AI system is intended as an AI-assisted research
prototype for turmeric adulteration screening and estimation.
Laboratory verification is recommended before practical deployment.